# Get to Know a Dataset: Downing Herbarium - Macquarie University

This notebook serves as a guided tour of the [Downing Herbarium - Macquarie University](https://registry.opendata.aws/downing-herbarium-mq) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

The [Downing Herbarium](https://www.mq.edu.au/about/facilities/museums-collections/the-downing-herbarium) at Macquarie University (Sydney, Australia) houses more than 5,000 bryophyte specimens (mosses, liverworts and hornworts from 142 families) and 11,000 vascular plant specimens, collected over more than 70 years (1947–2026). Specimens come from all Australian states, New Zealand, Antarctica and the subantarctic islands. With support from an Atlas of Living Australia (ALA) grant, the collection is being digitised. This dataset contains high-resolution images and specimen-level metadata for a representative subset of about 2,000 specimens, with at least one specimen of each species held in the collection.

<!-- DRAFT NOTE: the bucket layout below is our proposed layout (following Appendix A of the AWS onboarding handbook).
Confirm or update the bucket name, prefixes and file names once the data are uploaded to S3, then remove this note. -->

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The dataset is made up of two parts: **specimen images** and a **metadata table** that describes each specimen (taxonomy, collection date, locality, collector, etc.). The S3 bucket is organized as follows:

```
s3://downing-herbarium-mq/
├── README.txt                  # overview of the dataset
├── LICENSE.txt                 # CC BY 4.0
├── metadata/
│   └── specimens.csv           # one row per specimen, Darwin Core terms
└── data/
    └── jpeg/
        ├── bryophytes/         # mosses, liverworts and hornworts
        │   └── <catalogNumber>.jpg
        └── vascular-plants/
            └── <catalogNumber>.jpg
```

Each image file is named after the specimen's herbarium catalogue number (the `catalogNumber` column in `metadata/specimens.csv`), which links each image to its metadata record.

Full documentation for this dataset can be found at: https://figshare.com/s/6830f33a6cf8641a4aee

First we will import the Python libraries required throughout this notebook.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38.23
# pandas >= 2.2.0
# matplotlib >= 3.10.3
# pillow >= 11.0.0

# Import the libraries required for this notebook
# Built-ins
import io
# Installed libraries
import boto3, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from botocore import UNSIGNED
from botocore.config import Config

Next, we will define the location of our dataset, create our boto3 S3 client, and list the top level of our S3 bucket. At the top level we find the `README.txt` and `LICENSE.txt` files, alongside the `data/` and `metadata/` prefixes.

In [ ]:
# Location of the S3 bucket for this dataset
bucket = 'downing-herbarium-mq'  # DRAFT: confirm bucket name
region = 'ap-southeast-2'        # DRAFT: confirm bucket region

# Because this is a public bucket, we don't need to sign requests.
s3 = boto3.client('s3', region_name=region, config=Config(signature_version=UNSIGNED))

# Print the top-level prefixes and objects
response = s3.list_objects_v2(Bucket=bucket, Delimiter='/')
for item in response.get('CommonPrefixes', []):
    print(item['Prefix'])
for item in response.get('Contents', []):
    print(item['Key'])

The `README.txt` gives a short description of the dataset and its layout.

In [ ]:
# Read and print the README
with s3.get_object(Bucket=bucket, Key='README.txt')['Body'] as body:
    print(body.read().decode('utf-8'))

Images are split by plant group under `data/jpeg/`. Because the dataset holds a few thousand objects, we use a paginator to walk all of them, and then count the images and their total size in each group.

In [ ]:
# List the plant-group prefixes within 'data/jpeg/'
for item in s3.list_objects_v2(Bucket=bucket, Prefix='data/jpeg/', Delimiter='/')['CommonPrefixes']:
    print(item['Prefix'])

# Walk every image object and collect its key and size
paginator = s3.get_paginator('list_objects_v2')
images = pd.DataFrame(
    [{'key': obj['Key'], 'size_mb': obj['Size'] / 1e6}
     for page in paginator.paginate(Bucket=bucket, Prefix='data/jpeg/')
     for obj in page.get('Contents', [])
     if obj['Key'].lower().endswith('.jpg')]
)
images['group'] = images['key'].str.split('/').str[2]

print()
print(images.groupby('group')['size_mb'].agg(['count', 'sum', 'mean']).round(1))
print()
print(images['key'].head().to_string(index=False))

<!-- DRAFT NOTE: confirm the image format (JPEG vs TIFF), typical resolution / DPI, and whether images include a scale bar
and colour chart once the files are finalised, then remove this note. -->

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

The dataset uses two formats:

**1. JPEG specimen images (`data/jpeg/`).** Each herbarium sheet (vascular plants) or specimen packet (bryophytes) was photographed at high resolution. Each image typically shows the pressed specimen, its collection label, and the herbarium accession stamp/barcode. We chose JPEG because it is universally supported, and its compression keeps high-resolution images to a manageable size while keeping the detail needed for identification and label reading.

JPEG images can be read in Python with [Pillow](https://pillow.readthedocs.io/), [OpenCV](https://opencv.org/) or [scikit-image](https://scikit-image.org/), and in most deep-learning frameworks. Herbarium images are large, so we recommend making thumbnails (`Image.thumbnail()`) for browsing and only working at full resolution when you need fine detail, such as leaf venation, bryophyte leaf cells or label text.

**2. CSV specimen metadata (`metadata/specimens.csv`).** This is a single comma-separated table with one row per specimen. Column names follow the [Darwin Core](https://dwc.tdwg.org/terms/) standard, the biodiversity data standard used by the [Atlas of Living Australia](https://www.ala.org.au/), the [Australasian Virtual Herbarium](https://avh.ala.org.au/) and [GBIF](https://www.gbif.org/). Using Darwin Core means these records can be joined directly with millions of other occurrence records. Key columns include:

| Column | Description |
|---|---|
| `catalogNumber` | Downing Herbarium accession number; also the image file name |
| `scientificName`, `family`, `genus`, `phylum` | Taxonomic identification |
| `eventDate`, `year` | Date of collection |
| `recordedBy` | Collector(s) |
| `country`, `stateProvince`, `locality` | Where the specimen was collected |
| `decimalLatitude`, `decimalLongitude` | Collection coordinates (WGS84), where available |

CSV is easily read with pandas or Polars. At larger scale, it can be queried in place with SQL using [Amazon Athena](https://docs.aws.amazon.com/athena/latest/ug/what-is.html). For machine learning on the images (e.g. species identification or trait extraction), [Amazon SageMaker AI](https://aws.amazon.com/sagemaker/) can read the images directly from S3, and [Amazon Textract](https://aws.amazon.com/textract/) can be used to extract text from specimen labels.

### Q: Can you show us an example of downloading and loading data from your dataset?

We'll start by loading the specimen metadata from `s3://downing-herbarium-mq/metadata/specimens.csv` into a pandas DataFrame.

In [ ]:
# Load the specimen metadata table into a pandas DataFrame
metadata_key = 'metadata/specimens.csv'

with s3.get_object(Bucket=bucket, Key=metadata_key)['Body'] as body:
    specimens = pd.read_csv(io.BytesIO(body.read()), low_memory=False)

print(f'{len(specimens):,} specimen records with {specimens.shape[1]} columns')
specimens.head()

Let's look at a single specimen record. Empty fields are dropped so we only see what was recorded for that specimen.

In [ ]:
# Show all non-empty fields for the first specimen
specimens.iloc[0].dropna()

Image file names match the `catalogNumber` column, so we can link each record to its image by adding an `image_key` column. Then we load one image straight from S3 into memory with Pillow.

In [ ]:
# Link each metadata record to its image using the catalogue number in the file name
key_by_catalog = {key.rsplit('/', 1)[-1].rsplit('.', 1)[0]: key for key in images['key']}
specimens['image_key'] = specimens['catalogNumber'].astype(str).map(key_by_catalog)
print(f"{specimens['image_key'].notna().sum():,} of {len(specimens):,} records matched to an image")

def load_image(key, max_size=None):
    """Download an image from S3 and open it with Pillow, optionally shrinking it to fit max_size pixels."""
    with s3.get_object(Bucket=bucket, Key=key)['Body'] as body:
        img = Image.open(io.BytesIO(body.read()))
        img.load()
    if max_size:
        img.thumbnail((max_size, max_size))
    return img

# Load and display the first specimen at full resolution
record = specimens.dropna(subset=['image_key']).iloc[0]
img = load_image(record['image_key'])
print(f'Format: {img.format}, size: {img.size[0]} x {img.size[1]} px, mode: {img.mode}')

fig, ax = plt.subplots(figsize=(7, 10))
ax.imshow(img)
ax.set_title(f"{record['scientificName']}  ({record['catalogNumber']})", fontsize=12)
ax.axis('off')
plt.show()

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

First, the specimens themselves. Below is a gallery of a random selection of specimens, one from each of 12 different families, loaded as thumbnails to keep downloads small.

In [ ]:
# Pick one random imaged specimen from each of 12 random families
gallery = (
    specimens.dropna(subset=['image_key', 'family'])
    .groupby('family').sample(1, random_state=42)
    .sample(12, random_state=42)
)

fig, axes = plt.subplots(2, 6, figsize=(18, 8), facecolor='white')
for ax, (_, row) in zip(axes.flat, gallery.iterrows()):
    ax.imshow(load_image(row['image_key'], max_size=600))
    ax.set_title(f"{row['scientificName']}\n{row['family']}", fontsize=9)
    ax.axis('off')
fig.suptitle('Specimens from the Downing Herbarium', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

Next we look at the metadata. To compare the two halves of the collection, we add a `group` column that separates bryophytes (the phyla Bryophyta, Marchantiophyta and Anthocerotophyta) from vascular plants. We also make sure `year` is numeric.

In [ ]:
# Classify each specimen as a bryophyte or vascular plant from its phylum
BRYOPHYTE_PHYLA = ['Bryophyta', 'Marchantiophyta', 'Anthocerotophyta']
specimens['group'] = specimens['phylum'].isin(BRYOPHYTE_PHYLA).map({True: 'Bryophytes', False: 'Vascular plants'})

# Make sure collection year is numeric (unparseable values become NaN)
specimens['year'] = pd.to_numeric(specimens['year'], errors='coerce')

specimens.groupby('group').agg(
    specimens=('catalogNumber', 'count'),
    families=('family', 'nunique'),
    species=('scientificName', 'nunique'),
    earliest=('year', 'min'),
    latest=('year', 'max'),
)

The collection spans more than seven decades. How was collecting effort spread over that time?

In [ ]:
# Count specimens per collection year for each group
counts_by_year = specimens.dropna(subset=['year']).groupby(['year', 'group']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6), dpi=100, facecolor='white')
counts_by_year.plot(kind='bar', stacked=True, width=0.9, ax=ax, color=['#2e7d32', '#a5d6a7'], edgecolor='none')

ax.set_title('Specimens collected per year', fontsize=16, pad=20, fontweight='bold')
ax.set_xlabel('Collection year', fontsize=12, labelpad=10)
ax.set_ylabel('Number of specimens', fontsize=12, labelpad=10)
# Label every fifth year to keep the axis readable
ax.set_xticks(range(0, len(counts_by_year), 5))
ax.set_xticklabels(counts_by_year.index[::5].astype(int), rotation=45)
ax.grid(axis='y', linestyle='--', alpha=0.3, color='gray')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(title=None, frameon=False)
plt.tight_layout()
plt.show()

Where were the specimens collected? Plotting the coordinates of each georeferenced specimen shows the collection's geographic reach, from mainland Australia to New Zealand, the subantarctic islands and Antarctica. (For a map with coastlines, try [cartopy](https://scitools.org.uk/cartopy/) or [geopandas](https://geopandas.org/).)

In [ ]:
# Keep only georeferenced specimens
located = specimens.dropna(subset=['decimalLatitude', 'decimalLongitude'])
print(f'{len(located):,} of {len(specimens):,} specimens have coordinates')

fig, ax = plt.subplots(figsize=(10, 10), dpi=100, facecolor='white')
for (group, rows), color in zip(located.groupby('group'), ['#2e7d32', '#ef6c00']):
    ax.scatter(rows['decimalLongitude'], rows['decimalLatitude'], s=12, alpha=0.6, color=color, label=group)

ax.set_title('Collection localities', fontsize=16, pad=20, fontweight='bold')
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_aspect('equal')
ax.grid(True, linestyle='--', alpha=0.3, color='gray')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

Finally, which plant families are best represented in the collection?

In [ ]:
# The 15 families with the most specimens in each group
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=100, facecolor='white')
for ax, (group, rows), color in zip(axes, specimens.groupby('group'), ['#2e7d32', '#ef6c00']):
    rows['family'].value_counts().head(15).sort_values().plot(kind='barh', ax=ax, color=color)
    ax.set_title(group, fontsize=14, fontweight='bold')
    ax.set_xlabel('Number of specimens')
    ax.set_ylabel(None)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
fig.suptitle('Most represented families', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

<!-- DRAFT NOTE: this is a placeholder analysis. Replace it with a question from the herbarium team's own work if there is a
better one, and add a short interpretation of the results once the notebook has been run against the real data. -->

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**Is collecting effort evenly spread across regions and time, or is it biased?**

Herbarium collections are often used to study how species' distributions and traits have changed over time. However, specimens are rarely collected systematically, because collecting follows researchers, field trips and teaching programs. Before using a collection to answer ecological questions, it is important to understand these biases. Here we cross-tabulate specimens by decade and by state/region of collection to see where, and when, collecting effort was concentrated.

In [ ]:
# Count specimens by decade and state/region of collection
specimens['decade'] = (specimens['year'] // 10 * 10).astype('Int64')
region_by_decade = pd.crosstab(
    specimens['stateProvince'].fillna(specimens['country']).fillna('Unknown'),
    specimens['decade'],
)
region_by_decade = region_by_decade.loc[region_by_decade.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(12, 7), dpi=100, facecolor='white')
im = ax.imshow(region_by_decade, cmap='Greens', aspect='auto')
ax.set_xticks(range(len(region_by_decade.columns)), [f'{d}s' for d in region_by_decade.columns])
ax.set_yticks(range(len(region_by_decade.index)), region_by_decade.index)
# Write the count in each cell
for i in range(region_by_decade.shape[0]):
    for j in range(region_by_decade.shape[1]):
        count = region_by_decade.iat[i, j]
        if count:
            ax.text(j, i, count, ha='center', va='center', fontsize=8,
                    color='white' if count > region_by_decade.values.max() / 2 else 'black')
ax.set_title('Specimens by region and decade of collection', fontsize=16, pad=20, fontweight='bold')
fig.colorbar(im, ax=ax, label='Number of specimens')
plt.tight_layout()
plt.show()

# Share of specimens from the five best-sampled regions
top5_share = region_by_decade.sum(axis=1).head(5).sum() / region_by_decade.values.sum()
print(f'The five best-sampled regions account for {top5_share:.0%} of all specimens')

The heatmap shows where collecting effort was concentrated. Anyone using this collection to compare places or time periods should account for this uneven sampling, for example by comparing proportions instead of raw counts, by subsampling to equal effort, or by combining these records with other herbaria via the [Australasian Virtual Herbarium](https://avh.ala.org.au/).

<!-- DRAFT NOTE: describe what the heatmap actually shows (e.g. which regions/decades dominate and why, such as university field
trips or particular collectors) once the notebook has been run. -->

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

**Can plant traits and reproductive state be measured automatically from the specimen images, and have they changed over 70 years of a changing climate?**

Every herbarium specimen is a snapshot of a plant at a known place and time. In vascular plants, the images record leaf size and shape and whether the plant was flowering or fruiting when collected. In bryophytes, they record whether sporophytes (spore-bearing capsules) were present. Measured across the whole collection, these traits could show shifts in leaf size or in the timing of reproduction since the 1940s. Such shifts would be early signals of plants responding to a warming climate. Measuring these traits by hand for thousands of specimens is slow, which is where computer vision can help.

Some suggestions for getting started:

- **Start small.** Pick one well-represented family or genus (see the family chart above), so that you compare like with like.
- **Use pretrained models.** Vision models trained on natural-history images, such as [BioCLIP](https://imageomics.github.io/bioclip/) or general-purpose models like DINOv2, can provide image features or zero-shot labels (e.g. *flowering* vs *not flowering*) with little labelled data. Segmentation models can isolate leaves for measurement.
- **Calibrate measurements.** Use the scale bar on each image (where present) to convert pixel measurements into real-world sizes.
- **Read the labels.** OCR tools such as [Amazon Textract](https://aws.amazon.com/textract/) can extract text from the specimen labels. This can be used to check or add to the transcribed metadata, e.g. habitat notes.
- **Add climate data.** Join each specimen's coordinates and collection date to historical climate data, e.g. the [ERA5 reanalysis](https://registry.opendata.aws/ecmwf-era5/) on the Registry of Open Data, to test whether trait changes track temperature or rainfall.
- **Mind the sampling biases** described in the previous section, and consider combining these images with other digitised Australasian herbaria to increase sample sizes.

We'd love to hear from anyone who takes up this challenge.